## Matching distance 

This is the notebook where we will merge all the code to compute the matching distance

In [1]:
import gudhi
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np
import matplotlib
import matplotlib.patches as mpatches

from matplotlib import rc

# rc('font',**{'family':'sans-serif','sans-serif':['Helvetica']})
rc("font", **{"family": "serif", "serif": ["Times"]})
rc("text", usetex=True)


from math import cos, sin, pi, atan     
import itertools
from scipy.stats import qmc
import math

import numpy.linalg
import random as rnd

## for checking if in convex hull
import scipy.spatial
from scipy.optimize import linprog

import sys 
sys.path.append('../Code/')
import modules as mod
import examples as ex
import plot as ourplot
import csv
import bunny_smaller
import omega_pts as omega
# This reloads external modules every time a cell is run
# so we don't have to reboot kernel every time we modify classes.py

%load_ext autoreload
%autoreload 2

## 1. Loading an example and extracting the critical values in the correct format

In [2]:
critical_vals, nolub, M, N = ex.small_socg_ex_M_N()
print("Critical values: ", critical_vals)

Critical values:  [(parent module of (0.0, 0.0) is 0), (parent module of (6.0, 0.0) is 0), (parent module of (0.0, 6.0) is 0), (parent module of (4.0, 2.0) is 1), (parent module of (4.0, 4.0) is 1), (parent module of (8.0, 2.0) is 1), (parent module of (1.0, 5.0) is 1), (parent module of (1.0, 10.0) is 1), (parent module of (2.0, 5.0) is 1)]


In [3]:
# put the modules into the correct format (eg like in examples.py)
critval_coords = [] ## forgets the parent modules 
for i in range(len(critical_vals)):
    critval_coords.append(critical_vals[i].coords)
print(critval_coords)

[(0.0, 0.0), (6.0, 0.0), (0.0, 6.0), (4.0, 2.0), (4.0, 4.0), (8.0, 2.0), (1.0, 5.0), (1.0, 10.0), (2.0, 5.0)]


## 2. Identify the closures $\overline{C_M}$ and $\overline{C_N}$ 

In [4]:
lub = omega.closure_lub(critical_vals) ## computes the least upper bound CM and CN separately using the parent module information 
for i in range(len(lub)):
    print(lub[i])

## These are lists not sets so there are duplicates... maybe needs to be changed 

[(6.0, 0.0), (0.0, 6.0), (6.0, 0.0), (6.0, 6.0), (0.0, 6.0), (6.0, 6.0)]
[(4.0, 4.0), (8.0, 2.0), (4.0, 5.0), (4.0, 10.0), (4.0, 5.0), (4.0, 4.0), (8.0, 4.0), (4.0, 5.0), (4.0, 10.0), (4.0, 5.0), (8.0, 2.0), (8.0, 4.0), (8.0, 5.0), (8.0, 10.0), (8.0, 5.0), (4.0, 5.0), (4.0, 5.0), (8.0, 5.0), (1.0, 10.0), (2.0, 5.0), (4.0, 10.0), (4.0, 10.0), (8.0, 10.0), (1.0, 10.0), (2.0, 10.0), (4.0, 5.0), (4.0, 5.0), (8.0, 5.0), (2.0, 5.0), (2.0, 10.0)]


## 3. Compute the set of switch-points $\Omega=\Omega(M,N)$ using the algorithms 

In [5]:
### run all three allgoriths 3v1, 2v2 paired , 2v2 unpaired
### copied and pasted from socg_examples.ipynb


## For now: rename. Later: rewrite. 
critpts, crits_no_lub = critical_vals, nolub
makegraphyesno = False
title = 'title'

############################
if makegraphyesno:
    title = "M and N"
    dpi = 300

    ############ make the example
    fig, ax0 = plt.subplots()
    fig.set_dpi(dpi)
    fig.set_size_inches(6, 5)
    # fig.tight_layout

    fig.suptitle("Input rectangle modules M and N")
    for i in [0, 3, 6]:
        p1, p2, p3 = crits_no_lub[0 + i][0], crits_no_lub[1 + i][0], crits_no_lub[2 + i][0]
        p4 = np.array(p2) + np.array(p3) - np.array(p1)
        x = [p1[0], p2[0], p4[0], p3[0], p1[0]]
        y = [p1[1], p2[1], p4[1], p3[1], p1[1]]
        ax0.plot(x, y, "k--")

    for c in crits_no_lub:
        if c[1] == 0:
            ax0.plot(
                c[0][0],
                c[0][1],
                "o",
                color="black",
                label="black",
                markersize=4,
            )
        else:
            ax0.plot(
                c[0][0],
                c[0][1],
                "o",
                color="black",
                markersize=4,
            )
            ax0.plot(
                c[0][0],
                c[0][1],
                "o",
                color="orange",
                label="orange",
                markersize=3,
            )


    black_patch = mpatches.Patch(color="black", label="critical values from M")
    orange_patch = mpatches.Patch(color="orange", label="critical values from N")
    ax0.legend(
        handles=[black_patch, orange_patch],
        loc="upper right",
    )
    plt.show()


############# do computations
# algo 3v1 = Algorithm 3 in SoCG overleaf
# never has points at infinity, so we can plot all points
print("3v1")
omegapts_3v1, _, _ = omega.output_alg_3_1(critpts, "socg/" + title)
if makegraphyesno:
    ourplot.plot_both_c_o(crits_no_lub, omegapts_3v1, title, dpi, "3v1")

# algo 2v2 paired = Algorithm 5 in SoCG overleaf
print("2v2paired")
omegapts_2v2_p, _, _, is_inf_2v2p, orderedomegas2v2p = omega.output_alg_2_2_paired(
    critpts, "socg/" + title
)

if makegraphyesno:
    ourplot.plot_c_o_with_inf(crits_no_lub, orderedomegas, is_inf, title, dpi, "2v2 Paired")

# algo 2v2 unpaired = Algorithm 7 in SoCG overleaf
print("2v2unp")
omegapts_2v2_unp, _, _, is_inf_2v2unp_orig , orderedomegas_2v2unp = omega.output_alg_2_2_unpaired(
    critpts, "socg/" + title
)
if makegraphyesno:
    ourplot.plot_c_o_with_inf(
        crits_no_lub, orderedomegas2, is_inf2, title, dpi, "2v2 Unpaired"
    )

3v1


../Code/omega_pts.py:35: OptimizeWarning: A_eq does not appear to be of full row rank. To improve performance, check the problem formulation for redundant equality constraints.
  b_eq=b,
../Code/omega_pts.py:35: OptimizeWarning: Solving system with option 'cholesky':True failed. It is normal for this to happen occasionally, especially as the solution is approached. However, if you see this frequently, consider setting option 'cholesky' to False.
  b_eq=b,
../Code/omega_pts.py:35: OptimizeWarning: Solving system with option 'sym_pos':True failed. It is normal for this to happen occasionally, especially as the solution is approached. However, if you see this frequently, consider setting option 'sym_pos' to False.
  b_eq=b,
/Users/celiahacker/opt/anaconda3/envs/celia/lib/python3.7/site-packages/scipy/optimize/_linprog_ip.py:117: LinAlgWarning: Ill-conditioned matrix (rcond=2.77382e-18): result may not be accurate.
  return sp.linalg.solve(M, r, sym_pos=sym_pos)
/Users/celiahacker/opt/anac

**********
Computed 914 points (with duplicates)
Without duplicates 142
2v2paired
**********
Computed 133 points (with duplicates)
Without duplicates 44
2v2unp
**********
Computed 181 points (with duplicates)
Without duplicates 95


## 3.5: Now we should put all the 3 algos outputs together into one big omega, and deal with slopes vs points.

In [6]:
omega_points_collection = []
omega_slopes_collection = []
is_inf_2v2unp = is_inf_2v2unp_orig  

for n in range(len(omegapts_3v1)):
    omega_points_collection.append(omegapts_3v1[n])

omega_points_collection += omegapts_3v1[0:] 


for s in is_inf_2v2p[1:]:
    m1 = s 
    m2 = 1
    omega_slopes_collection.append((m1, m2))
    
    
orderedomegas_2v2unp_copy = orderedomegas_2v2unp.copy()

orderedomegas_2v2unp_copy.pop(0)

print(orderedomegas_2v2unp_copy)


is_inf_2v2unp = is_inf_2v2unp[1:]
for thing in range(len(orderedomegas_2v2unp_copy)-1):
    if is_inf_2v2unp[thing] == None: 
        omega_points_collection.append(orderedomegas_2v2unp_copy[thing])

    else:
        omega_slopes_collection.append(orderedomegas_2v2unp_copy[thing])
        print(orderedomegas_2v2unp_copy[thing]) 

        


[(3.0, 6.0), (-6.0, -2.0), (2.0, -6.0), (7.333333333333333, 2.0), (2.0, 5.333333333333333), (2.0, 8.666666666666666), (-2.0, -6.0), (-2.0, -4.0), (-2.0, -5.0), (-2.0, -10.0), (5.0, 5.0), (7.0, 2.0), (7.0, 2.5), (7.0, 5.0), (3.0, 5.0), (3.0, 7.5), (3.0, 10.0), (6.0, 5.0), (4.0, 2.0), (4.0, 1.0), (2.0, 5.5), (2.0, 8.0), (8.0, 2.0), (8.0, 1.0), (4.0, 5.5), (4.0, 8.0), (2.0, 1.0), (1.0, 8.0), (1.3333333333333333, 5.0), (0.0, -5.0), (1.3333333333333333, 10.0), (-4.0, -0.0), (0.0, -10.0), (2.6666666666666665, 5.0), (2.6666666666666665, 8.333333333333334), (2.6666666666666665, 10.0), (-8.0, -0.0), (2.0, 4.0), (7.0, 4.0), (5.0, 5.5), (5.0, 8.0), (2.0, 2.0), (7.0, 5.5), (7.0, 8.0), (-5.0, -4.0), (3.5, 8.0), (-4.0, -4.0), (5.333333333333333, 4.0), (6.666666666666667, 3.3333333333333335), (8.0, 6.0), (5.333333333333333, 4.666666666666667), (8.0, 16.0), (5.333333333333333, 8.0), (6.666666666666667, 4.0), (6.666666666666667, 7.333333333333333), (5.333333333333333, 5.0), (8.0, 15.0), (5.333333333333

## 4. For each  line $L$ with positive slope that:

- passes through pairs of points in $\overline{C_M}\cup \overline{C_N}$, or
- passes through one point in $\overline{C_M}\cup \overline{C_N}$ and one point in ${\Omega}$,   or 
- passes through  one point of $\overline{C_M}\cup \overline{C_N}\cup{\Omega}$ and has diagonal slope

do 

a. Compute the persistence diagrams $dgm(M^L)$, $dgm(N^L)$ of the restrictions of $M$ and $N$ to that line $L$;


b. Compute their weighted bottleneck distance $\hat m^L d_B(dgm(M^L),dgm(N^L))$


#### We need clarification in 4. 

In [7]:
lub0  = list(set(lub[0]))
lub1 = list(set(lub[1]))
closureMN = critval_coords + lub0 + lub1

pointstoiterateover = closureMN + omega_points_collection  ### do we need to add again the critical values here 
sloppyslopestoiterateover = omega_slopes_collection

Lines = []
Wb  = []


## Lines through pairs of points 

for x in range(len(pointstoiterateover)):
    for y in range(x, len(pointstoiterateover)):
        L = mod.line_through_2points(pointstoiterateover[x], pointstoiterateover[y]) ## here is the issue with none type error
        if L.positive == True:
            Lines.append(L)
            w = mod.compute_weighted_bottleneck(M,N,L)
            Wb.append(w)

In [8]:
## Lines through a point and with a given slope

for x in range(len(closureMN)): 
    for y in range(len(sloppyslopestoiterateover)):
        L = mod.Line(sloppyslopestoiterateover[y], closureMN[x])
        if L.positive == True:
            Lines.append(L)
            w = mod.compute_weighted_bottleneck(M,N,L)
            Wb.append(w)
            


# Lines of slope 1 through one point 
            
for x in range(len(pointstoiterateover)):
    L = mod.Line((1,1), pointstoiterateover[x])
    if L.positive == True:
        Lines.append(L)
        w = mod.compute_weighted_bottleneck(M,N,L)
        Wb.append(w) 
            
        

## 5. Output the maximum of the weighted bottleneck distances computed in (b). 

In [9]:
matching_dist = max(Wb)
idx = Wb.index(matching_dist)

print('The matching distance is {}'.format(matching_dist))
print('Matching disstance is attained at the line with slope {} and intercept {}'.format(Lines[idx].m, Lines[idx].b))


The matching distance is 3.0
Matching disstance is attained at the line with slope (1.0, 1.0) and intercept (0.0, 0.0)


### 6. checking modules with one rectangle 

$$ \min ( \max ( \min \frac{b_i-a_i}{2} \min \frac{d_i-c_i}{2}  ), max( ||c-a||_\infty , ||d-b||_\infty   )  $$